In [ ]:
import os
# API key is read from the environment / .env at runtime (see README). Never hardcode keys.

In [ ]:
"""
PA Policy Parameter Extraction Pipeline — Integrated with Document Reducer.

Architecture:
  1. Document Reducer (pre-computed): Docling structured output → algorithmic
     section extraction → focused text per (filename, brand) pair.
  2. This pipeline: reads reduced text → Gemini extraction → validation → CSV.

The reducer handles all document structures locally (heading match, table search,
cross-reference following). The LLM only does the hard reasoning: step counting,
AND/OR logic, phototherapy classification.

Inputs:
  work/reduced/   — Reducer output (primary)
  work/md_output/    — Full markdown (fallback if reducer output missing)
  PA_Business_Rules.xlsx — Submission list

Outputs:
  work/result.csv           — 79-row extraction results
  work/_extraction_log.json — Detailed extraction log

Run: GEMINI_API_KEY=your-key python pipeline_integrated.py
"""
import os
import json
import csv
import re
from pathlib import Path
from google import genai
from google.genai import types
import openpyxl

# === CONFIG ===
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
if not GEMINI_API_KEY:
    raise RuntimeError("Set GEMINI_API_KEY env var.")

REDUCED_DIR = os.environ.get("REDUCED_DIR", "work/reduced")
MD_DIR = os.environ.get("MD_DIR", "work/md_output")
EXCEL_PATH = os.environ.get("EXCEL_PATH", "PA_Business_Rules.xlsx")
OUTPUT_CSV = os.environ.get("OUTPUT_CSV", "work/result.csv")

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL_ID = "gemini-2.5-flash"

# === EXTRACTION PROMPT ===
SYSTEM_PROMPT = """You are a pharmaceutical prior authorization policy analyst. You extract structured parameters from PA policy documents for the Plaque Psoriasis (PsO) indication.

You will receive a policy document (or relevant excerpt) and a target brand name. Extract the 12 parameters defined below for that brand's PsO coverage ONLY.

=== CRITICAL DISTINCTION: "NA" vs "No" ===

- "NA" = The information COULD NOT BE FOUND in the policy document. The policy simply does not mention this parameter for PsO. Data is missing/absent.
- "No" = The policy EXPLICITLY STATES there is no restriction or requirement.

If a parameter is not mentioned anywhere in the PsO section or universal criteria, output "NA" — NOT "No".

SCOPE OF THIS RULE — two cases:
- ABSENT-DATA fields (quantity limits, specialist types, durations, TB test): if not mentioned, output "NA".
- STEP-TYPE fields (number of steps through brands, number of steps through generic, phototherapy): if the policy HAS PsO step criteria but this particular step is simply not required, output "No" — the requirement is absent, not the data. Reserve "NA" for these step-type fields ONLY when the policy has NO PsO step criteria for the brand at all (a true negative — the indication isn't covered, or there is no PA block).

=== CRITICAL RULES ===

INDICATION FILTERING:
- Extract ONLY for Plaque Psoriasis (PsO). Ignore all other indications.
- If the policy separates "moderate-to-severe" and "severe" PsO, extract ONLY moderate-to-severe criteria.
- SPECIALIST TYPES: Only capture specialists listed in the PsO section. Do NOT include Gastroenterologist or other specialists from CD/UC sections.
- STEP THERAPY: Only count steps from the PsO section and universal/general criteria. Do NOT count steps from other indications.
- SELF-REFERENCING: If the target brand appears as one of several options in a step therapy requirement, exclude it from the option list but still count the step — the patient would use one of the other options. Only exclude the entire step if the target brand is the sole option (a drug cannot be the only prerequisite to access itself). This can occur due to document redlining artifacts.

UNIVERSAL vs INDICATION-SPECIFIC CRITERIA:
- Some policies have "General Requirements" or "Universal Criteria" that apply across all indications, PLUS separate PsO-specific criteria.
- These two sets combine with AND — the patient must satisfy BOTH.
- When counting steps, take the UNION of universal steps and PsO-specific steps.

STEP COUNTING — LEAST RESTRICTIVE PATH:
- OR statements → take the path with FEWER total steps (least restrictive).
- AND statements → patient must complete ALL (union).
- "Tried Drug A OR Drug B" = 1 step, NOT 2.
- "Tried Drug A AND Drug B" = 2 steps.
- "must have tried ONE of the following: [list of 8 drugs]" = 1 step, not 8.
- "must have tried TWO of the following: [list]" = 2 steps.

STEP CLASSIFICATION:
- Branded/biologic: Named biologic drugs, TNF inhibitors, IL-inhibitors, targeted synthetic drugs, biosimilars. A preferred ustekinumab or adalimumab product counts as a branded step. If a step references a drug CLASS (e.g., TNF antagonists) and the target drug belongs to that class, that class-level step counts as branded.
- Generic: Topical agents, conventional systemics (methotrexate, cyclosporine, acitretin), non-biologic drugs.
- UNNAMED STEPS DEFAULT TO GENERIC: If a required step does NOT name any specific biologic/targeted drug (e.g., "trial THREE preferred products" with no products named), it counts as a GENERIC step — steps with no biologic specification count as generic. Do NOT assume unnamed "preferred products" are biologics even in a biologic policy; classify by what the text states, not by what the products probably are.
- Phototherapy: EXCLUDED from both counts — captured separately.

PHOTOTHERAPY (a step-type field — treat exactly like the step counts):
- Phototherapy INCLUDES PUVA (psoralen + UVA) and UVB. Combine universal and PsO-specific phototherapy language with AND, then assess.
- "Yes" = phototherapy is a MANDATORY required step (AND, with no alternative path). If the policy provides any alternative path that does not require phototherapy (an OR option, or an explicit exception that removes the requirement for some patients), then phototherapy is not mandatory → "No".
- "No" = the policy HAS PsO step criteria, but phototherapy is NOT a required step — whether it is absent from the criteria entirely, or appears only as one option inside an OR. Not-a-required-step means "No", NOT "NA".
- "NA" = ONLY when the policy establishes NO PsO step criteria for this brand at all (a true negative).

CONNECTOR DEFAULT: When two step statements are listed with NO explicit AND/OR connector between them, treat them as an OR relationship (take the least-restrictive path). Per the reference example, separate unconnected criteria statements default to OR, not AND.

=== PARAMETER DEFINITIONS ===

1. AGE — Output the age threshold the POLICY states. Rules, in order:
   (a) If the policy prints an explicit NUMERIC age (e.g. "must be >=18 years", "patients 6 years and older", "18 years of age"), output that number (e.g. ">=18", ">=6").
   (b) If the policy describes the population only with a word like "adult"/"adults" (or "pediatric", "children") WITHOUT printing a numeric age, that is NOT a numeric threshold. It means the policy is deferring to the FDA-approved population without stating a number → output the literal string "FDA labelled age". Do NOT convert "adult" to ">=18". Example: a policy saying "treatment of adult patients with moderate-to-severe plaque psoriasis", with no number anywhere in the criteria, → "FDA labelled age", NOT ">=18".
   (c) If the policy gives no number but only says the drug is used/indicated per its FDA label (e.g. "for FDA labeled indications", "per FDA-approved age") → output "FDA labelled age".
   (d) In (b) and (c), output the literal words "FDA labelled age" — do NOT substitute an age you happen to know for the drug.
   (e) If age is not mentioned at all, or PsO/this drug's age appears only in incidental text (a step-therapy carve-out for another group, or an FDA-informational blurb in a policy that does not actually cover this drug for PsO) → "NA". If the policy explicitly states no age restriction → "No".
   (f) If the policy lists two age groups WITH numbers, output the YOUNGEST number. (If one drug in a multi-drug list carries a bracketed number, e.g. "[>=6 years]", use that number for THAT drug; a drug in the same list with no bracketed number → "FDA labelled age".)
   The distinction between (a) and (b)/(c) depends ENTIRELY on whether the policy text prints an actual number for this drug — never on outside knowledge of the drug's approved age. The word "adult" alone is NOT a number.

2. STEP THERAPY REQUIREMENTS — All step therapy language for PsO. "NA" if none.

3. NUMBER OF STEPS THROUGH BRANDS — Integer as string, "No" if criteria exist but no branded steps, "NA" if no criteria.

4. NUMBER OF STEPS THROUGH GENERIC — Same logic.

5. STEP THROUGH PHOTOTHERAPY — "Yes"=mandatory required step (AND). "No"=policy has PsO step criteria but phototherapy is not a required step (absent, or only inside an OR). "NA"=ONLY when no PsO step criteria exist for the brand at all.

6. TB TEST REQUIRED — "Yes" if a TB test/screening is a REQUIRED condition for approval (e.g. "documented negative TB test required prior to initiation"). "No" if the policy explicitly states TB testing is not required. "NA" if TB is not mentioned, or appears ONLY as an exclusion / contraindication / warning (a mention is not a requirement).

7. QUANTITY LIMITS — Only explicitly labeled "quantity limit". Not "dosage"/"dosing". "NA" if not mentioned.

8. SPECIALIST TYPES — PsO only. "NA" if not mentioned.

9. INITIAL AUTHORIZATION DURATION (MONTHS) — Convert weeks/days/years to months. "NA" if not mentioned.

10. REAUTHORIZATION DURATION (MONTHS) — Same. "NA" if not mentioned.

11. REAUTHORIZATION REQUIRED — "Yes"/"No"/"NA". If reauth duration or requirements is non-NA, must be "Yes".

12. REAUTHORIZATION REQUIREMENTS — Text or "NA".

=== FEW-SHOT EXAMPLES ===

EXAMPLE 1 — Phototherapy in OR, no age mentioned
{
  "age": "NA",
  "step_therapy_requirements": "inadequate response to phototherapy or methotrexate/cyclosporine/acitretin",
  "number_of_steps_through_brands": "No",
  "number_of_steps_through_generic": "1",
  "step_through_phototherapy": "No",
  "tb_test_required": "Yes",
  "quantity_limits": "NA",
  "specialist_types": "NA",
  "initial_authorization_duration": "12",
  "reauthorization_duration": "NA",
  "reauthorization_required": "No",
  "reauthorization_requirements": "NA"
}

EXAMPLE 2 — Branded + generic steps with AND (policy prints "must be 18 years or older" → numeric)
{
  "age": ">=18",
  "step_therapy_requirements": "ONE topical AND TWO biologics/targeted synthetics",
  "number_of_steps_through_brands": "2",
  "number_of_steps_through_generic": "1",
  "step_through_phototherapy": "No",
  "tb_test_required": "Yes",
  "quantity_limits": "NA",
  "specialist_types": "Dermatologist",
  "initial_authorization_duration": "12",
  "reauthorization_duration": "12",
  "reauthorization_required": "Yes",
  "reauthorization_requirements": "Positive clinical response: reduction in BSA or improvement in symptoms"
}

EXAMPLE 3 — OR logic, least restrictive path
{
  "age": ">=6",
  "step_therapy_requirements": "(topical + methotrexate) OR (targeted immunomodulator)",
  "number_of_steps_through_brands": "No",
  "number_of_steps_through_generic": "2",
  "step_through_phototherapy": "No",
  "tb_test_required": "Yes",
  "quantity_limits": "NA",
  "specialist_types": "Dermatologist",
  "initial_authorization_duration": "12",
  "reauthorization_duration": "12",
  "reauthorization_required": "Yes",
  "reauthorization_requirements": "Positive clinical response AND not combined with another targeted immunomodulator"
}

=== PsO BRAND REFERENCE ===

BRANDED/BIOLOGIC: Amjevita, Avsola, Bimzelx, Cimzia, Cosentyx, Cyltezo, Enbrel, Humira, Hyrimoz, Idacio, Ilumya, Inflectra, Hulio, Otezla, Remicade, Renflexis, Siliq, Skyrizi, Sotyktu, Stelara, Taltz, Tremfya, Vtama, Yuflyma, Yusimry, Zoryve, Wezlana, Selarsdi, Yesintek, Psychiva/Quallent, Steqeyma, Otulfi

GENERIC/CONVENTIONAL: Acitretin, Cyclosporine, Methotrexate, topical corticosteroids, vitamin D analogs, calcineurin inhibitors, tazarotene, coal tar, anthralin

=== OUTPUT FORMAT ===

Return a JSON object with exactly the 12 fields. No extra fields or commentary."""


# === LOAD SUBMISSIONS ===
def load_submissions():
    wb = openpyxl.load_workbook(EXCEL_PATH, data_only=True)
    ws = wb['Submissions']
    submissions = []
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, values_only=True):
        filename, brand = row[0], row[1]
        if not filename or not brand:
            continue
        submissions.append({"filename": str(filename).strip(), "brand": str(brand).strip()})
    wb.close()
    return submissions


# === LOAD CONTENT ===
def load_content(filename, brand):
    """
    Load pre-reduced content for a (filename, brand) pair.
    Priority 1: Reducer output in REDUCED_DIR (algorithmically extracted)
    Priority 2: Full markdown in MD_DIR (fallback)
    """
    stem = Path(filename).stem

    # Priority 1: Reducer output (brand-specific). Sanitize the brand for the filename
    # the same way the reducer does, so a brand containing '/' or spaces (e.g.
    # "Psychiva/Quallent") still resolves instead of silently falling through.
    safe_brand = re.sub(r'[^A-Za-z0-9._-]', '_', brand)
    candidates = [
        Path(REDUCED_DIR) / f"{stem}__{brand}.txt",
        Path(REDUCED_DIR) / f"{stem}__{safe_brand}.txt",
    ]
    for reduced_path in candidates:
        if reduced_path.exists():
            text = reduced_path.read_text(encoding="utf-8")
            if text.strip():
                return text, "reduced"

    # Priority 2: Full markdown
    md_path = Path(MD_DIR) / f"{stem}.md"
    if md_path.exists():
        return md_path.read_text(encoding="utf-8"), "markdown-fallback"

    return None, "missing"


# === PARSE JSON ===
def parse_json_response(text):
    """Parse JSON from model response, handling markdown fences and preamble."""
    if text is None:
        raise ValueError("Response text is None")

    cleaned = text.strip()

    # Find JSON block inside markdown fences anywhere in response
    fence_match = re.search(r'```(?:json)?\s*\n(.*?)\n```', cleaned, re.DOTALL)
    if fence_match:
        cleaned = fence_match.group(1).strip()
    # Or find first { ... } block
    elif not cleaned.startswith("{"):
        brace_start = cleaned.find("{")
        brace_end = cleaned.rfind("}")
        if brace_start != -1 and brace_end != -1:
            cleaned = cleaned[brace_start:brace_end + 1]

    return json.loads(cleaned)


# === GEMINI CALL HELPER ===
def call_gemini(contents, system_instruction=None, json_output=False, max_retries=3):
    """Generic Gemini call with retry logic."""
    config_params = {
        "temperature": 0.1,
        "thinking_config": types.ThinkingConfig(thinking_budget=-1),
    }
    if system_instruction:
        config_params["system_instruction"] = system_instruction
    if json_output:
        config_params["response_mime_type"] = "application/json"

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=contents,
                config=types.GenerateContentConfig(**config_params),
            )

            if not response.candidates:
                raise ValueError(f"No candidates returned: {getattr(response, 'prompt_feedback', 'unknown')}")

            candidate = response.candidates[0]
            finish_reason = getattr(candidate, "finish_reason", None)
            # Accept normal completion. SDK may represent this as enum FinishReason.STOP,
            # the name "STOP", or the int 1 — treat all of these as success. Only treat
            # an explicitly non-stop reason (MAX_TOKENS, SAFETY, RECITATION) as a failure.
            if finish_reason is not None:
                fr = str(finish_reason).upper()
                fr_name = str(getattr(finish_reason, "name", "")).upper()
                is_stop = ("STOP" in fr) or (fr_name == "STOP") or (fr in ("1", "FINISHREASON.STOP"))
                bad = any(x in fr or x in fr_name for x in ("MAX_TOKENS", "SAFETY", "RECITATION", "BLOCKLIST", "PROHIBITED"))
                if bad or (not is_stop and fr not in ("NONE", "0", "FINISH_REASON_UNSPECIFIED")):
                    raise ValueError(f"Stopped: {finish_reason}")

            return response.text, None

        except Exception as e:
            print(f"    Error (attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                continue

    return None, "max_retries_exceeded"


# === PARAMETER EXTRACTION ===
def extract_parameters(policy_text, brand):
    """Extract 12 PA parameters from policy text."""
    user_prompt = f"""Extract the 12 PA parameters for the brand "{brand}" on the Plaque Psoriasis (PsO) indication from the following policy document.

POLICY DOCUMENT:
{policy_text}"""

    # Retry the whole call+parse: a malformed-JSON response is recoverable on a
    # fresh attempt, so don't treat the first parse failure as terminal.
    last_error = None
    for attempt in range(3):
        text, error = call_gemini(user_prompt, system_instruction=SYSTEM_PROMPT, json_output=True)
        if error:
            last_error = error
            continue
        try:
            return parse_json_response(text), None
        except (json.JSONDecodeError, ValueError) as e:
            last_error = f"JSON parse error: {e}"
            print(f"    Parse failed (attempt {attempt+1}/3): {e}; retrying")
            continue

    return None, last_error


# === VALIDATE AND NORMALIZE ===
def validate_and_normalize(params, brand):
    # Age: keep the model's output as-is (the prompt already applies the rubric:
    # policy number -> that number; FDA-label-without-number -> "FDA labelled age").
    # Do NOT resolve "FDA labelled age" to a number — the rubric requires the literal
    # string. Only normalize the spelling to match the rubric exactly.
    age = str(params.get("age", "NA")).strip()
    age_l = age.lower()
    if "fda" in age_l:
        params["age"] = "FDA labelled age"
    elif re.fullmatch(r"(adults?|pediatric|children|paediatric)\b.*", age_l) and not re.search(r"\d", age):
        # Backstop: model emitted a population word with no number, ignoring the prompt.
        # Per the rubric this is a deferral to the FDA label, not a numeric threshold.
        params["age"] = "FDA labelled age"

    # Binary fields
    for key in ["tb_test_required", "reauthorization_required"]:
        val = str(params.get(key, "NA")).strip().lower()
        if val in ("yes", "y", "true"):
            params[key] = "Yes"
        elif val in ("no", "n", "false"):
            params[key] = "No"
        else:
            params[key] = "NA"

    # Step counts — extract leading digits, normalize 0 to "No". (Done before
    # durations so we can detect whether the policy establishes PsO coverage.)
    for key in ["number_of_steps_through_brands", "number_of_steps_through_generic"]:
        val = params.get(key, "NA")
        val_str = str(val).strip().lower()
        if val_str in ("na", "n/a", "none", ""):
            params[key] = "NA"
        elif val_str in ("no", "0"):
            params[key] = "No"
        elif isinstance(val, (int, float)):
            params[key] = "No" if int(val) == 0 else str(int(val))
        else:
            cleaned = str(val).strip()
            digit_match = re.match(r'(\d+)', cleaned)
            if digit_match:
                num = int(digit_match.group(1))
                params[key] = "No" if num == 0 else str(num)

    # Phototherapy
    photo = str(params.get("step_through_phototherapy", "NA")).strip().lower()
    if photo in ("yes", "y"):
        params["step_through_phototherapy"] = "Yes"
    elif photo in ("no", "n"):
        params["step_through_phototherapy"] = "No"
    else:
        params["step_through_phototherapy"] = "NA"

    # Does the policy establish PsO coverage criteria for this brand? If ANY core
    # PsO parameter is non-NA, coverage exists (it is not a true-negative row).
    # Per the Business Rules, when PsO coverage exists the duration fields must be
    # either a number OR "Unspecified" — never blank/NA. We only fall back to "NA"
    # on true-negative rows where the policy doesn't cover this brand for PsO at all.
    # Coverage is judged from SUBSTANTIVE criteria only (steps, phototherapy, TB,
    # step-therapy text, specialist). Age and durations are NOT coverage evidence —
    # a policy can mention an age in passing (e.g. an FDA blurb) without covering the
    # drug for PsO, so including age here would wrongly mark true-negatives as covered.
    pso_signals = [
        params.get("number_of_steps_through_brands", "NA"),
        params.get("number_of_steps_through_generic", "NA"),
        params.get("step_through_phototherapy", "NA"),
        params.get("tb_test_required", "NA"),
        str(params.get("step_therapy_requirements", "NA")),
        str(params.get("specialist_types", "NA")),
        str(params.get("quantity_limits", "NA")),
    ]
    pso_coverage = any(str(s).strip().upper() not in ("NA", "N/A", "NONE", "") for s in pso_signals)

    # No true-negative override here — an all-NA substantive profile could be a
    # genuine non-coverage OR an extraction miss on a covered policy. Forcing
    # age/durations to NA on a miss would silently destroy real data. The duration
    # coverage-gate below handles Unspecified-vs-NA conservatively instead.

    # Durations — extract leading digits. Preserve "Unspecified" as a valid value.
    # Rubric: if PsO coverage exists, a missing/unstated duration is "Unspecified",
    # not "NA". reauth_duration is only required to be a number/Unspecified when
    # reauthorization is required.
    def normalize_duration(val, coverage_exists):
        if isinstance(val, (int, float)):
            return str(int(val))
        s = str(val).strip()
        sl = s.lower()
        if sl == "unspecified":
            return "Unspecified"
        if sl in ("", "none", "null", "na", "n/a"):
            return "Unspecified" if coverage_exists else "NA"
        m = re.match(r'(\d+)', s)
        if m:
            return m.group(1)
        # Non-empty but no number (e.g. "varies", "per plan") and coverage exists
        return "Unspecified" if coverage_exists else "NA"

    # Reauthorization requirements: normalize to NA if absent, else keep the text.
    rr_raw = params.get("reauthorization_requirements", "NA")
    rr_is_na = str(rr_raw).strip().lower() in ("na", "n/a", "none", "null", "")
    params["reauthorization_requirements"] = "NA" if rr_is_na else str(rr_raw).strip()

    # Cross-field reauth (rubric): if the raw reauth DURATION or the reauth
    # REQUIREMENTS is non-NA, reauthorization_required must be "Yes". Evaluate this
    # BEFORE normalizing the duration, so a flip to "Yes" correctly forces the
    # duration to a number-or-"Unspecified" below (never leaving Yes + NA duration).
    raw_reauth_dur = params.get("reauthorization_duration", "")
    raw_dur_is_na = str(raw_reauth_dur).strip().lower() in ("na", "n/a", "none", "null", "")
    if (not raw_dur_is_na) or (not rr_is_na):
        params["reauthorization_required"] = "Yes"

    # Durations — preserve "Unspecified"; per rubric, when coverage/reauth applies,
    # a missing duration is "Unspecified", not "NA".
    params["initial_authorization_duration"] = normalize_duration(
        params.get("initial_authorization_duration", ""), pso_coverage)

    reauth_required_final = str(params.get("reauthorization_required", "NA")).strip().lower() == "yes"
    params["reauthorization_duration"] = normalize_duration(
        params.get("reauthorization_duration", ""), reauth_required_final)

    return params


# === CSV COLUMNS ===
CSV_COLUMNS = [
    ("Filename", lambda row, p: row["filename"]),
    ("Brand", lambda row, p: row["brand"]),
    ("Age", lambda row, p: p.get("age", "")),
    ("Step Therapy Requirements Documented in Policy", lambda row, p: p.get("step_therapy_requirements", "")),
    ("Number of Steps through Brands", lambda row, p: p.get("number_of_steps_through_brands", "")),
    ("Number of Steps through Generic", lambda row, p: p.get("number_of_steps_through_generic", "")),
    ("Step through-Phototherapy", lambda row, p: p.get("step_through_phototherapy", "")),
    ("TB Test required", lambda row, p: p.get("tb_test_required", "")),
    ("Quantity Limits", lambda row, p: p.get("quantity_limits", "")),
    ("Specialist Types", lambda row, p: p.get("specialist_types", "")),
    ("Initial Authorization Duration(in-months)", lambda row, p: p.get("initial_authorization_duration", "")),
    ("Reauthorization Duration(in-months)", lambda row, p: p.get("reauthorization_duration", "")),
    ("Reauthorization Required", lambda row, p: p.get("reauthorization_required", "")),
    ("Reauthorization Requirements Documented in Policy", lambda row, p: p.get("reauthorization_requirements", "")),
    ("Access Score", lambda row, p: p.get("access_score", "")),
]


# === MAIN PIPELINE ===
def run_pipeline():
    submissions = load_submissions()
    print(f"Loaded {len(submissions)} submission rows")
    print(f"Reduced dir: {REDUCED_DIR}")
    print(f"Markdown dir: {MD_DIR}")

    if not submissions:
        print("ERROR: No valid submissions found.")
        return

    # Verify directories exist
    if not Path(REDUCED_DIR).exists():
        print(f"WARNING: {REDUCED_DIR} not found. All files will use markdown fallback.")
    if not Path(MD_DIR).exists():
        print(f"ERROR: {MD_DIR} not found. Cannot proceed.")
        return

    results = []
    errors = []
    source_counts = {"reduced": 0, "markdown-fallback": 0, "missing": 0}

    for i, row in enumerate(submissions):
        filename = row["filename"]
        brand = row["brand"]
        print(f"[{i+1}/{len(submissions)}] {filename} | {brand}")

        # Load content (reducer output preferred, markdown fallback)
        policy_text, source = load_content(filename, brand)
        if policy_text is None:
            print(f"  ERROR: content not found ({source})")
            errors.append({"filename": filename, "brand": brand, "error": "file_not_found"})
            results.append({"row": row, "params": {}})
            source_counts["missing"] += 1
            continue

        if not policy_text.strip():
            print(f"  ERROR: empty content ({source})")
            errors.append({"filename": filename, "brand": brand, "error": "empty_file"})
            results.append({"row": row, "params": {}})
            source_counts["empty"] = source_counts.get("empty", 0) + 1
            continue

        source_counts[source] = source_counts.get(source, 0) + 1
        print(f"  Source: {source} ({len(policy_text):,} chars)")

        # Extract parameters
        params, error = extract_parameters(policy_text, brand)
        if error:
            print(f"  EXTRACTION ERROR: {error}")
            errors.append({"filename": filename, "brand": brand, "error": error})
            results.append({"row": row, "params": {}})
            continue

        params = validate_and_normalize(params, brand)
        params["access_score"] = ""

        results.append({"row": row, "params": params})
        print(f"  OK: age={params.get('age')}, brands={params.get('number_of_steps_through_brands')}, "
              f"generic={params.get('number_of_steps_through_generic')}, "
              f"auth={params.get('initial_authorization_duration')}")

    # Write CSV
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([col[0] for col in CSV_COLUMNS])
        for r in results:
            writer.writerow([col[1](r["row"], r["params"]) for col in CSV_COLUMNS])

    # Summary
    print(f"\n{'='*60}")
    print(f"Done. {len(results) - len(errors)}/{len(results)} succeeded.")
    print(f"Sources: {source_counts}")
    print(f"Errors: {len(errors)}")
    if errors:
        for e in errors:
            print(f"  {e['filename']} | {e['brand']}: {e['error']}")
    print(f"Output: {OUTPUT_CSV}")

    # Log
    log_path = Path(OUTPUT_CSV).parent / "_extraction_log.json"
    with open(log_path, "w", encoding="utf-8") as f:
        json.dump(
            [{"filename": r["row"]["filename"], "brand": r["row"]["brand"], "params": r["params"]} for r in results],
            f, indent=2,
        )


if __name__ == "__main__":
    run_pipeline()